# CLIFFGUARD — the labelled suites, and the second annotation axis

**Runtime: T4 GPU is enough.** `Runtime -> Change runtime type -> T4 GPU`, then
`Run all`. A100 or L4 is faster but changes nothing.

This is a **new** notebook. It does not replace `colab_run.ipynb` or
`colab_round2.ipynb`, and it writes to `lab-*` run directories, so nothing from
either earlier round is touched.

## What it adds

The manuscript can say *the model's decision changed* and cannot say *whether
that was good*, because its prompts carry no harmfulness annotation and its
grader has one axis. This notebook fixes both, separately.

| axis | where the labels come from | what it settles |
|---|---|---|
| **prompt**: harmful / benign | published suites — HarmBench, AdvBench, StrongREJECT, XSTest, OR-Bench | *should* the model have helped? |
| **completion**: refusal / compliance / deflection / disclaimer / unclear | a 7B judge, five-way, first-token argmax | what did it actually *do*? |

Crossed, they give the object the whole project has been missing:

|            | refusal | compliance | deflection | disclaimer | degenerate |
|---|---|---|---|---|---|
| **harmful** | withheld (desired) | **SAFETY FAILURE** | partial withhold | non-answer | capability failure |
| **benign**  | **OVER-REFUSAL** | utility (desired) | soft over-refusal | capability failure | capability failure |

The two bold cells are regressions and they point in opposite directions, so
they are never summed. An unlabelled corpus reports them as one number.

**The asymmetry is real and is not hidden.** The prompt axis is externally
supplied and not this project's opinion. The completion axis is a second model
judge, and it is not human ground truth. That remains the largest open gap, and
nothing below closes it — what it does close is the *conflation* of refusal with
deflection and with capability disclaimer, which is a different and previously
uncontrolled error.

## It survives its own runtime ending

Every stage is a step in a journal on Drive, and the journal is the point:

* a step that finished in an earlier session is **skipped**;
* a step that finished with **different arguments** is re-run, and says so;
* a step that finished but whose output is **gone** is re-run;
* inside a step, the ladder and the graders cache **per scheme**, so a rung that
  finished is durable the moment it finishes;
* near the end of a session the pipeline **declines to start** a long step,
  stopping between steps instead of being killed inside one.

So: if the runtime dies, reconnect and run the pipeline cell again. Nothing that
finished is repeated.

## 0 — Environment

Installs only what Colab lacks. `numpy` is deliberately **not** pinned: forcing
`numpy<2` breaks Colab's preinstalled torch through an ABI mismatch.

`CLIFFGUARD_ARTIFACTS` is set to a Drive path. That is what makes run
directories durable — written relative to the clone they would live in
`/content` and vanish with the session, after the GPU time that produced them
has been spent.

In [ ]:
import os, sys, json, pathlib, subprocess, platform

IN_COLAB   = "google.colab" in sys.modules
REPO_URL   = "https://github.com/parnish007/CLIFFGUARD.git"
BRANCH     = "main"
REPO_DIR   = pathlib.Path("/content/CLIFFGUARD") if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/cliffguard")

if IN_COLAB:
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] NOT MOUNTED:", exc)
        print("        Progress will be lost when this runtime ends. Stop and fix this.")
        DRIVE_ROOT = pathlib.Path("/content/cliffguard_local")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1",
                        "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard",
                        f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.44", "accelerate", "bitsandbytes",
                    "datasets", "sentencepiece", "protobuf"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

# Run directories go to Drive, not to the clone. Read by cliffguard.eval.storage
# at import time, so it must be set before anything imports it.
ARTIFACTS = DRIVE_ROOT / "artifacts"
os.environ["CLIFFGUARD_ARTIFACTS"] = str(ARTIFACTS)
(ARTIFACTS / "runs").mkdir(parents=True, exist_ok=True)

CACHE_ROOT = DRIVE_ROOT / "cache_labelled"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

import torch
VRAM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9
           if torch.cuda.is_available() else 0.0)
print(f"repo      : {REPO_DIR}")
print(f"drive     : {DRIVE_ROOT}")
print(f"artifacts : {ARTIFACTS}")
print(f"python    : {platform.python_version()}   torch {torch.__version__}")
print(f"gpu       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}"
      f"   {VRAM_GB:.1f} GB")
if not torch.cuda.is_available():
    print("!! no GPU: Runtime -> Change runtime type -> T4 GPU")

## 1 — Configuration

Set here, before the preflight, because the preflight checks these values: a
tokenizer that cannot separate the five labels is a fact about *this* judge, and
learning it after the config cell would mean learning it after the download.

`N_PER_CLASS` is **per harmfulness class**, so a suite carrying both classes
yields up to `2 × N_PER_CLASS` prompts and a single-class suite yields
`N_PER_CLASS`. The loader interleaves the classes before truncating, which
matters: the suite files are class-ordered, and truncating first would silently
give a run with zero harmful prompts and a table that looks fine.

Bit-widths are the paper's seven rungs. The two collapsed rungs cost little and
are where the degeneracy gate earns its place, so they stay.

In [ ]:
MODELS = [
    ("qwen3b", "Qwen/Qwen2.5-3B-Instruct"),
    ("phi35",  "microsoft/Phi-3.5-mini-instruct"),
]

# XSTest first: it is the only suite carrying BOTH classes, so it is the only
# one that gives a full 2x2 from a single run. The others are single-class and
# are read against it rather than alone.
SUITES = ["xstest", "harmbench", "or-bench-hard"]

JUDGE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_4BIT    = VRAM_GB < 20          # a T4 cannot hold 7B in fp16
N_PER_CLASS   = 200
BITS          = [8, 7, 6, 5, 4, 3, 2]
MAX_NEW       = 48
BATCH         = 16 if VRAM_GB >= 20 else 8
JUDGE_BATCH   = 8 if VRAM_GB >= 20 else 4

# How long this session may run before the pipeline stops BETWEEN steps rather
# than being killed inside one. Free Colab reclaims at ~4 h, Pro at ~12 h; set
# it slightly under whichever you have.
DEADLINE_HOURS = 3.5 if VRAM_GB < 20 else 11.0

RUNS_ROOT = ARTIFACTS / "runs"
JOURNAL   = DRIVE_ROOT / "journal_labelled.json"

print(f"models   : {[m for m, _ in MODELS]}")
print(f"suites   : {SUITES}")
print(f"judge    : {JUDGE_MODEL}  (4-bit: {JUDGE_4BIT})")
print(f"prompts  : up to {N_PER_CLASS} per class per suite")
print(f"rungs    : {BITS}")
print(f"journal  : {JOURNAL}")
print(f"deadline : {DEADLINE_HOURS} h")

## 2 — PREFLIGHT

Seconds, on CPU, before any download or GPU time. It imports every symbol the
run uses, checks the signatures that matter, exercises the resumable runner
end-to-end on a toy step, and — the expensive one to get wrong — verifies that
the five taxonomy labels have **distinct first tokens** under the judge's
tokenizer. First-token argmax over labels that share a first piece is not a
five-way choice at all; it produces verdicts that look fine and mean nothing.

`PREFLIGHT OK` means nothing below can die on an import, a signature, or a
tokenizer surprise. If it fails, **stop**.

In [ ]:
import numpy as np, tempfile, pathlib as _pl

failures = []
def check(label, fn):
    try:
        fn(); print(f"  ok    {label}")
    except Exception as exc:
        failures.append(f"{label}: {type(exc).__name__}: {exc}")
        print(f"  FAIL  {label}: {type(exc).__name__}: {exc}")

print("imports")
from scripts.colab_pipeline import Pipeline, Step, latest_run, argv_fingerprint
from scripts.classify_completion_taxonomy import (
    LABELS, CLASS_OF, TAXONOMY_TEMPLATE, label_first_token_ids, resolve, gate_mask)
from scripts.analyse_matrix import CLASSES, WITHHELD, contingency, paired, _rate
from scripts.analyse_labelled import HARMFUL, BENIGN, strata
from scripts.run_behavioural_ladder import is_degenerate, DEGENERACY_NLL_MULTIPLE
from scripts.run_local_ladder import write_json_atomic, read_json_cache, load_labelled_prompts
from cliffguard.eval.storage import ARTIFACTS_ROOT
print("  ok    every symbol the run uses")

check("artifacts root points at Drive",
      lambda: (_ for _ in ()).throw(AssertionError(f"{ARTIFACTS_ROOT} is not under {DRIVE_ROOT}"))
              if str(DRIVE_ROOT) not in str(ARTIFACTS_ROOT) else None)

def _pipeline_roundtrip():
    with tempfile.TemporaryDirectory() as d:
        d = _pl.Path(d); marker = d / "ran.txt"
        for _ in range(2):
            p = Pipeline(journal=d / "j.json", log_dir=d / "logs", cwd=d)
            p.add(Step("t", [sys.executable, "-c", f"open(r'{marker}','a').write('x')"]))
            assert p.run() == 0
        assert marker.read_text() == "x", "a finished step ran twice"
check("resumable runner skips finished work", _pipeline_roundtrip)

def _taxonomy_tokens():
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
    ids = label_first_token_ids(tok)          # raises if they collide
    assert len(set(ids)) == len(LABELS)
    print(f"        {dict(zip(LABELS, ids))}")
check("five taxonomy labels have distinct first tokens", _taxonomy_tokens)

def _gate_and_resolve():
    texts = ["I cannot help with that.", "@@@@ @@@@ @@@@ @@@@ @@@@ @@@@", ""]
    nll = np.array([1.0, 1.0, 1.0]); ok = gate_mask(texts, nll, 5.0)
    assert ok[0] and not ok[1] and not ok[2], f"gate misfired: {ok}"
    out = resolve(["REFUSE", "COMPLY", "COMPLY"], ok)
    assert list(out) == ["refusal", "degenerate", "degenerate"], list(out)
check("degeneracy is decided before the judge's verdict", _gate_and_resolve)

def _matrix_cells():
    harm = np.array([HARMFUL] * 4 + [BENIGN] * 4)
    labels = {"FP16": np.array(["refusal", "refusal", "compliance", "compliance",
                                "compliance", "compliance", "refusal", "refusal"]),
              "RTN_4B": np.array(["compliance", "refusal", "compliance", "refusal",
                                  "refusal", "compliance", "refusal", "compliance"])}
    rows = paired(labels, harm, "strict")
    r = rows[0]
    assert r["safety_lost"] == 1, r          # harmful, refusal -> compliance
    assert r["safety_recovered"] == 1, r     # harmful, compliance -> refusal
    assert r["over_refusal"] == 1, r         # benign, compliance -> refusal
    assert r["over_refusal_relaxed"] == 1, r # benign, refusal -> compliance
check("the four matrix cells are counted in the right corners", _matrix_cells)

def _suites_present_or_downloadable():
    import scripts.download_eval_suites as dl
    assert set(SUITES) <= set(dl.SUITES), f"unknown suite: {set(SUITES) - set(dl.SUITES)}"
check("every configured suite is one the downloader knows", _suites_present_or_downloadable)

print()
if failures:
    raise SystemExit("PREFLIGHT FAILED:\n  " + "\n  ".join(failures))
print("PREFLIGHT OK")

## 3 — Suites

Assembled by the repository's own downloader, from the canonical sources.
HarmBench, AdvBench and StrongREJECT come from their GitHub CSVs rather than the
HuggingFace mirrors, which are gated. XSTest is split into its harmful and
benign halves by the suite's own `type` field, with an assertion that the split
is non-empty on both sides.

The result is deterministic and hashed into `MANIFEST.json`, so a re-download
that changes anything is visible rather than silent.

In [ ]:
subprocess.run([sys.executable, "scripts/download_eval_suites.py", "--download"],
               check=True, cwd=REPO_DIR)
manifest = json.loads((REPO_DIR / "data/eval_suites/MANIFEST.json").read_text())
total = 0
for s in manifest["suites"]:
    total += s["n"]
    print(f"  {s['suite']:16s} n={s['n']:5d}  "
          f"harmful={s['counts']['harmful']:5d} benign={s['counts']['benign']:5d}  "
          f"{s['sha256_16']}  {s['provenance']}")
print(f"\n{total} prompts across {len(manifest['suites'])} suites")
if manifest["failed"]:
    print("FAILED:", manifest["failed"])

## 4 — The pipeline

**This is the cell to re-run after a disconnect.** It builds every step, skips
what is done, and resumes what is not.

Per (model, suite): the ladder, then two graders on its output. Both graders run
because they answer different questions and the comparison between them is
itself a result — the three-way grader is the one the manuscript's numbers come
from, and the five-way one is what shows how much of its `REFUSE` class was
never refusal.

The grading steps take the run directory as an argument, and a run directory is
named with the timestamp at which it was created, so they cannot be written down
in advance. They are built lazily, resolved by label at the moment they start;
that is how a resumed session finds the ladder output an earlier session made.

In [ ]:
from scripts.colab_pipeline import Pipeline, Step, latest_run

def suite_path(suite):  return f"data/eval_suites/{suite}.jsonl"
def label_of(model, suite):  return f"lab-{model}-{suite}"

pipe = Pipeline(journal=JOURNAL, log_dir=DRIVE_ROOT / "logs_labelled",
                cwd=REPO_DIR, deadline_hours=DEADLINE_HOURS)

pipe.add(Step("suites",
              [sys.executable, "scripts/download_eval_suites.py", "--download"],
              produces=REPO_DIR / "data/eval_suites/MANIFEST.json",
              estimated_minutes=3, timeout_s=900))

for model_key, model_id in MODELS:
    for suite in SUITES:
        label = label_of(model_key, suite)
        # Caches are per model AND per suite: two suites at the same n and token
        # budget would otherwise collide on the same cache filename and the
        # second would silently read the first's completions.
        cache = CACHE_ROOT / f"{model_key}_{suite}"

        pipe.add(Step(
            f"ladder-{label}",
            [sys.executable, "scripts/run_behavioural_ladder.py",
             "--model", model_id, "--prompts", suite_path(suite),
             "--n", str(N_PER_CLASS), "--bits", *map(str, BITS),
             "--max-new-tokens", str(MAX_NEW), "--batch-size", str(BATCH),
             "--cache", str(cache), "--label", label],
            estimated_minutes=55, timeout_s=4 * 3600))

        pipe.add(Step(
            f"judge3-{label}",
            (lambda lab=label: [
                sys.executable, "scripts/classify_completions_judge.py",
                str(latest_run(RUNS_ROOT, lab)),
                "--judge-model", JUDGE_MODEL, "--batch-size", str(JUDGE_BATCH),
                *(["--judge-4bit"] if JUDGE_4BIT else [])]),
            produces=(lambda lab=label: latest_run(RUNS_ROOT, lab)
                      / "results" / "judge_classification.json"),
            estimated_minutes=25, timeout_s=3 * 3600))

        pipe.add(Step(
            f"taxonomy-{label}",
            (lambda lab=label: [
                sys.executable, "scripts/classify_completion_taxonomy.py",
                str(latest_run(RUNS_ROOT, lab)),
                "--judge-model", JUDGE_MODEL, "--batch-size", str(JUDGE_BATCH),
                *(["--judge-4bit"] if JUDGE_4BIT else [])]),
            produces=(lambda lab=label: latest_run(RUNS_ROOT, lab)
                      / "results" / "completion_taxonomy.json"),
            estimated_minutes=25, timeout_s=3 * 3600))

pipe.add(Step("analyse-labelled",
              [sys.executable, "scripts/analyse_labelled.py",
               "--runs", str(RUNS_ROOT), "--include", "*lab-*",
               "--out", str(DRIVE_ROOT / "labelled_stats.json")],
              estimated_minutes=2, timeout_s=1800))

pipe.add(Step("analyse-matrix",
              [sys.executable, "scripts/analyse_matrix.py",
               "--runs", str(RUNS_ROOT), "--include", "*lab-*",
               "--out", str(DRIVE_ROOT / "matrix_stats.json")],
              estimated_minutes=2, timeout_s=1800))

failed = pipe.run()
print(f"\n{failed} step(s) failed" if failed else "\nall steps complete")

## 5 — Results

Reads the two analysis files and prints the matrix. Nothing is computed here:
every number comes from a repository script, so there is no second
implementation to drift out of sync with the one that is published.

In [ ]:
for name in ("matrix_stats.json", "labelled_stats.json"):
    path = DRIVE_ROOT / name
    if not path.exists():
        print(f"[{name}] not written yet"); continue
    data = json.loads(path.read_text())
    print(f"\n{'#' * 78}\n# {name}\n{'#' * 78}")
    for key, block in data.items():
        print(f"\n=== {key}  ({block.get('model')}, {block.get('corpus')}) ===")
        if "contingency" in block:
            fp16 = block["contingency"]["FP16"]
            cols = list(next(iter(fp16.values())).keys())
            print(f"{'FP16':10s}" + "".join(f"{c[:10]:>12s}" for c in cols))
            for pc, row in fp16.items():
                print(f"{pc:10s}" + "".join(f"{row[c]:12d}" for c in cols))
            for reading, rows in block["paired"].items():
                print(f"\n  withheld = {reading}")
                for r in rows:
                    print(f"    {r['scheme']:9s} safety {r['safety_lost']:4d}/"
                          f"{r['safety_recovered']:<4d} p={r['safety_p_holm']:.3f}   "
                          f"over-refusal {r['over_refusal']:4d}/"
                          f"{r['over_refusal_relaxed']:<4d} p={r['over_refusal_p_holm']:.3f}")
        else:
            for r in block.get("rows", []):
                print(f"    {r['scheme']:9s} lost={r['safety_lost']:4d} "
                      f"over={r['over_refusal']:4d}")

## 6 — Take it home

Everything already lives on Drive and is durable. This packs the small files —
manifests, results, analysis JSON — into one archive for downloading; the
per-scheme activation arrays are left behind because they are large and are
regenerable from the caches.

In [ ]:
import shutil, zipfile, time

stamp = time.strftime("%Y%m%d-%H%M%S")
bundle = DRIVE_ROOT / f"cliffguard_labelled_{stamp}.zip"
KEEP = (".json", ".md", ".log")

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for root in (RUNS_ROOT, DRIVE_ROOT / "logs_labelled"):
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and p.suffix in KEEP:
                z.write(p, p.relative_to(DRIVE_ROOT))
    for name in ("matrix_stats.json", "labelled_stats.json", "journal_labelled.json"):
        if (DRIVE_ROOT / name).exists():
            z.write(DRIVE_ROOT / name, name)

print(f"{bundle}  ({bundle.stat().st_size / 1e6:.1f} MB)")
pipe.report()
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(bundle))
    except Exception as exc:
        print("download failed; the archive is on Drive:", exc)